# Hw2.1: Pandas - Birth Rate Analysis

Analyzing birth rate coefficients in Ukrainian regions (1950-2019)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load data from Wikipedia

In [ ]:
url = 'https://uk.wikipedia.org/wiki/%D0%9D%D0%B0%D1%80%D0%BE%D0%B4%D0%BD%D0%BE%D1%81%D1%82%D1%8C_%D0%B2_%D0%A3%D0%BA%D1%80%D0%B0%D1%97%D0%BD%D1%96'

try:
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    tables = pd.read_html(StringIO(response.text), decimal=',', thousands=' ')
    df = tables[1].copy()
    print('Data loaded from Wikipedia successfully')
except Exception as e:
    print(f'Wikipedia unavailable ({e}), using sample data')
    df = pd.DataFrame({
        'Region': ['Kyiv', 'Lviv', 'Kharkiv', 'Odesa', 'Zaporizhia', 'Dnipro', 'Donetsk'],
        '2010': [10.5, 9.2, 9.8, 9.1, 10.3, 9.5, 9.9],
        '2014': [12.5, 11.2, 11.8, 11.1, 12.3, 11.5, 11.9],
        '2019': [9.5, 8.2, 8.8, 8.1, 9.3, 8.5, 8.9]
    })

print(f'Shape: {df.shape}')
print(f'\nFirst rows:')
df.head()

## 2. Replace dashes with NaN and convert types

In [ ]:
df = df.replace('—', np.nan)
df = df.replace('–', np.nan)
df = df.replace('-', np.nan)
print('Replaced dashes with NaN')

In [ ]:
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Data types after conversion:')
print(df.dtypes)

## 3. Analyze missing values

In [ ]:
missing_pct = (df.isnull().sum() / len(df)) * 100
print('Missing % by column:')
print(missing_pct[missing_pct > 0] if (missing_pct > 0).any() else 'No missing values')

## 4. Remove national data and fill missing values

In [ ]:
print(f'Before removing last row: {df.shape}')
df = df[:-1]
print(f'After removing last row: {df.shape}')

In [ ]:
for col in df.columns[1:]:
    if df[col].dtype in ['float64', 'int64']:
        df[col].fillna(df[col].mean(), inplace=True)

print('Missing values after fillna:', df.isnull().sum().sum())

## 5. Analysis: Regions above average in 2019

In [ ]:
col_2019 = df.columns[-1]
avg_2019 = df[col_2019].mean()
print(f'2019 Average: {avg_2019:.2f}')
print(f'\nRegions above average:')
above_avg = df[df[col_2019] > avg_2019][[df.columns[0], col_2019]]
print(above_avg)

## 6. Highest birth rate in 2014

In [ ]:
col_2014 = df.columns[-2] if len(df.columns) > 2 else df.columns[1]
max_idx = df[col_2014].idxmax()
max_region = df.loc[max_idx, df.columns[0]]
max_rate = df.loc[max_idx, col_2014]
print(f'Highest birth rate in 2014:')
print(f'  Region: {max_region}')
print(f'  Rate: {max_rate:.2f}')

## 7. Visualization 1: Bar chart by region

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sorted_df = df.sort_values(col_2019)
colors = ['#e74c3c' if x > avg_2019 else '#3498db' for x in sorted_df[col_2019]]
ax.barh(sorted_df[df.columns[0]], sorted_df[col_2019], color=colors, edgecolor='black', linewidth=1.5)
ax.axvline(avg_2019, color='green', linestyle='--', linewidth=2, label=f'Average: {avg_2019:.2f}')
ax.set_title('Birth Rate by Region (2019)', fontsize=14, fontweight='bold')
ax.set_xlabel('Birth Rate per 1000')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Visualization 2: Histogram distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df[col_2019], bins=6, color='#2ecc71', edgecolor='black', alpha=0.7)
ax.axvline(avg_2019, color='#e74c3c', linestyle='--', linewidth=2, label=f'Average: {avg_2019:.2f}')
ax.set_title('Distribution of Birth Rates (2019)', fontsize=14, fontweight='bold')
ax.set_xlabel('Birth Rate')
ax.set_ylabel('Number of regions')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Visualization 3: Line chart trends

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
numeric_cols = df.columns[1:].tolist()
for idx, row in df.iterrows():
    ax.plot(numeric_cols, df.loc[idx, numeric_cols].values, marker='o', label=row[df.columns[0]], alpha=0.7, linewidth=2)
ax.set_title('Birth Rate Trends by Region', fontsize=14, fontweight='bold')
ax.set_ylabel('Birth Rate')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Visualization 4: Box plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
df_numeric = df[numeric_cols]
bp = ax.boxplot([df_numeric[col].values for col in numeric_cols], labels=numeric_cols, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#f39c12')
    patch.set_alpha(0.7)
ax.set_title('Birth Rate Variations Across Years', fontsize=14, fontweight='bold')
ax.set_ylabel('Birth Rate')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 11. Visualization 5: Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
heatmap_data = df.set_index(df.columns[0])[numeric_cols]
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Birth Rate'})
ax.set_title('Birth Rate Heatmap by Region and Year', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print('Analysis complete!')
print(f'Dataset shape: {df.shape}')
print(f'Regions analyzed: {df.shape[0]}')
print(f'Years covered: {len(numeric_cols)}')
print(f'Average 2019 birth rate: {avg_2019:.2f}')
print(f'Regions above average: {len(above_avg)}')